# **1. Perkenalan Dataset**

Dataset yang digunakan adalah **Titanic Dataset**.

- **Sumber**: https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv
- **Deskripsi**: Dataset berisi informasi penumpang kapal Titanic dengan 891 sampel dan berbagai fitur seperti kelas tiket, jenis kelamin, usia, dan tarif.
- **Tujuan**: Memprediksi apakah seorang penumpang selamat (Survived=1) atau tidak (Survived=0).

# **2. Import Library**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
import os

print('Libraries imported successfully')

# **3. Memuat Dataset**

In [ ]:
url = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'
df = pd.read_csv(url)

os.makedirs('../titanic_raw', exist_ok=True)
df.to_csv('../titanic_raw/titanic_raw.csv', index=False)

print(f'Shape: {df.shape}')
df.head()

# **4. Exploratory Data Analysis (EDA)**

In [ ]:
print('=== Dataset Info ===')
df.info()
print('\n=== Statistik Deskriptif ===')
df.describe()

In [ ]:
print('=== Missing Values ===')
print(df.isnull().sum())
print(f'\nJumlah duplikat: {df.duplicated().sum()}')

In [ ]:
print('=== Distribusi Target (Survived) ===')
print(df['Survived'].value_counts())

plt.figure(figsize=(6, 4))
df['Survived'].value_counts().plot(kind='bar', color=['steelblue', 'orange'])
plt.title('Distribusi Target (Survived)')
plt.xlabel('Survived')
plt.ylabel('Jumlah')
plt.xticks([0, 1], ['Tidak Selamat', 'Selamat'], rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
feature_cols = ['Pclass', 'Age', 'SibSp', 'Parch', 'Fare']
df[feature_cols].hist(figsize=(10, 6), bins=15)
plt.suptitle('Distribusi Fitur Numerik')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(df[feature_cols].corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Heatmap Korelasi Fitur')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 4))
for i, col in enumerate(feature_cols):
    plt.subplot(1, 5, i+1)
    sns.boxplot(y=df[col])
    plt.title(col[:10])
plt.suptitle('Boxplot Fitur (Deteksi Outlier)')
plt.tight_layout()
plt.show()

# **5. Data Preprocessing**

In [ ]:
# 5.1 Pilih kolom yang relevan
df_clean = df[['Survived', 'Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']].copy()
print(f'Shape awal: {df_clean.shape}')

In [ ]:
# 5.2 Tangani missing values
df_clean['Age'].fillna(df_clean['Age'].median(), inplace=True)
df_clean['Embarked'].fillna(df_clean['Embarked'].mode()[0], inplace=True)
df_clean = df_clean.drop_duplicates().dropna()
print(f'Setelah handle missing values & duplikat: {df_clean.shape}')

In [ ]:
# 5.3 Encoding fitur kategorikal
le = LabelEncoder()
df_clean['Sex'] = le.fit_transform(df_clean['Sex'])
df_clean['Embarked'] = le.fit_transform(df_clean['Embarked'])
print('Label encoding selesai')
print(df_clean.head())

In [ ]:
# 5.4 Standarisasi fitur
feature_cols = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']
X = df_clean[feature_cols]
y = df_clean['Survived']

scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=feature_cols)

print('Statistik setelah standarisasi:')
X_scaled.describe().round(3)

In [ ]:
# 5.5 Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

In [ ]:
# 5.6 Simpan hasil preprocessing
os.makedirs('../titanic_preprocessing', exist_ok=True)

train_df = X_train.copy()
train_df['Survived'] = y_train.values
test_df = X_test.copy()
test_df['Survived'] = y_test.values

train_df.to_csv('../titanic_preprocessing/titanic_train.csv', index=False)
test_df.to_csv('../titanic_preprocessing/titanic_test.csv', index=False)

print('Dataset preprocessing tersimpan di titanic_preprocessing/')
print(f'Train: {train_df.shape}, Test: {test_df.shape}')